# 03. Approach 2: Deep Learning with Transfer Learning
### Avocado Ripening Stage Detection & Classification System
**Architecture:** MobileNetV2 (Pretrained ImageNet Backbone) + Custom 4-Class Classification Head  
**Optimization:** Cross-Entropy Loss, Adam Optimizer, Early Stopping

---
## Objective
Implement a lightweight, edge-deployable deep learning model for automated sorting conveyors:
1. **Transfer Learning:** Leverage general visual primitives (edges, gradients, textures) pretrained on ImageNet.
2. **Regularization & Augmentation:** Apply stochastic rotations, flips, and color jitter to prevent overfitting.
3. **Training & Validation:** Optimize with early stopping and record epoch loss/accuracy dynamics.


In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLASSES,
    DEVICE,
    SPLIT_MANIFEST_PATH,
    DL_MODEL_PATH,
    EXPERIMENTS_OUTPUTS_DIR,
)
from src.preprocessing import load_split_manifest
from src.dataset import create_dataloaders
from src.dl_model import build_avocado_classifier, load_dl_model
from src.training import train_model
from src.evaluation import evaluate_deep_learning

print(f"Execution Device: {DEVICE}")


## 1. Dataset Loading & DataLoaders
Building PyTorch DataLoaders with ImageNet normalization and training augmentations.


In [ ]:
manifest_df = load_split_manifest(SPLIT_MANIFEST_PATH)
train_loader, val_loader, test_loader = create_dataloaders(manifest_df, batch_size=16)

print(f"DataLoaders prepared: Train ({len(train_loader.dataset)}), Val ({len(val_loader.dataset)}), Test ({len(test_loader.dataset)})")


## 2. Model Architecture: MobileNetV2
MobileNetV2 uses depthwise separable convolutions with inverted residual blocks, maintaining high representational capacity with an ultra-compact memory footprint (~9MB).


In [ ]:
model = build_avocado_classifier(num_classes=4, pretrained=True)
print(model.classifier)


## 3. Model Training & Early Stopping
Training the classification head while tracking validation loss.


In [ ]:
# Run training loop
metadata = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=10,
    lr=0.001,
    patience=3,
    device=DEVICE,
    checkpoint_path=DL_MODEL_PATH,
)
print("Training completed. Metadata:", metadata)


## 4. Evaluation on Held-Out Test Set
Evaluating the best restored MobileNetV2 checkpoint on the test partition.


In [ ]:
test_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)
report, cm, latency_ms = evaluate_deep_learning(test_df, checkpoint_path=DL_MODEL_PATH, device=DEVICE)

print(f"MobileNetV2 Test Accuracy: {report['accuracy']*100:.2f}%")
print(f"MobileNetV2 Macro F1: {report['macro avg']['f1-score']:.4f}")
print(f"MobileNetV2 Latency: {latency_ms:.2f} ms")


## Conclusion
- MobileNetV2 achieves high classification performance on complex visual boundaries.
- Low latency makes it well-suited for high-throughput conveyor lines.
- Proceed to `04_model_evaluation_and_demo.ipynb` for unified comparative benchmarking.
